# Calibrating SLIP's difficulty dial against measured landscapes

Thomas et al. (2022) tune synthetic Potts landscapes to a target *epistatic
horizon*, the mutation count at which adaptive single mutants stop combining
into adaptive multi-mutants,

$$K_{\text{epi}} = 1 - 2\bar{s}_+ / \bar{\epsilon}_{+,+}$$

They report $K_{\text{epi}} = 166$ for an untuned Potts model on a protein of
length 118, and sweep their tuning over $2^1$ to $2^{10}$.

They never report the value for a landscape that was measured, because their
pipeline contains none. Complete empirical libraries supply it, which turns a
free difficulty dial into a quantity with a target.

Produced by `scripts/measure_epistatic_horizon.py`.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))
pd.set_option("display.width", 200)
ROOT = Path.cwd().parents[1] / "notebooks" / "iclr" / "output_260822_horizon"
horizon = pd.read_csv(ROOT / "epistatic_horizon.csv")

## Measured landscapes

In [2]:
measured = horizon[horizon.arm == "measured"].sort_values("k_epi")
measured[["dataset", "n_sites", "s_plus", "eps_plus", "k_epi", "n_adaptive_pairs"]].round(3).to_string(index=False)

'             dataset  n_sites  s_plus  eps_plus  k_epi  n_adaptive_pairs\n                DHFR        3   0.116     0.048 -3.827             316.0\n               ParD3        3   0.006    -0.445  1.025               4.0\n            RhlA_RLs       11   0.061    -2.060  1.060              14.0\n                 TEV        4   0.027    -0.137  1.399             111.0\n              TrpB3D        3   0.076    -0.205  1.740               6.0\n              TrpB3G        3   0.136    -0.365  1.744               3.0\n                  T7        3   0.248    -0.595  1.833              66.0\n        RhlA_product       11   0.731    -1.672  1.874             356.0\n               TrpB4        4   0.099    -0.143  2.385              27.0\nCreiLOV_fluorescence       15   0.057    -0.075  2.514             100.0\n         eqFP611_red       13   0.035    -0.043  2.609              66.0\n   CR9114_H1_binding       16   0.285    -0.268  3.128              78.0\n       RhlA_fraction       11   0.999

In [3]:
finite = measured[np.isfinite(measured.k_epi)]
print(f"finite K_epi on {len(finite)} of {len(measured)} landscapes")
print(f"median {finite.k_epi.median():.2f}   IQR {finite.k_epi.quantile(.25):.2f} to {finite.k_epi.quantile(.75):.2f}")
print(f"SLIP untuned Potts: 166      SLIP tuning sweep: 2 to 1024")

finite K_epi on 16 of 29 landscapes
median 2.13   IQR 1.65 to 3.41
SLIP untuned Potts: 166      SLIP tuning sweep: 2 to 1024


## Three findings SLIP could not reach

**Real landscapes sit at the bottom edge of the tuning sweep.** A median horizon
near two means that combining a second adaptive mutation already cancels the
additive gain. An untuned Potts model at 166 is roughly eighty times too linear,
and most of the swept range is far smoother than any measured library.

**The metric is undefined on many real assays.** Thirteen of twenty-nine
landscapes have no pair whose two singles are both beneficial, so the average
that defines the horizon has nothing to average over. Enzyme libraries where
every single mutation is deleterious are the common case, not an edge case.

**It is undefined on an additive surrogate too**, for the opposite reason. With
no epistasis there is no crossover to find.

In [4]:
additive = horizon[horizon.arm == "additive"]
print(f"additive rows {len(additive)}, finite K_epi {int(np.isfinite(additive.k_epi).sum())}, "
      f"median |eps| {additive.eps_plus.abs().median():.2e}")
print(f"landscapes with no adaptive pair: {int((measured.n_adaptive_pairs == 0).sum())} of {len(measured)}")

additive rows 16, finite K_epi 0, median |eps| 2.17e-17
landscapes with no adaptive pair: 13 of 29


## Where the posterior lands

The posterior reaches a median horizon of 3.27 against 2.13 measured, so it is
the right order of magnitude and still slightly too linear. That agrees with the
independent topology result, where it generates 14 local optima against 67
measured. Two unrelated metrics point at the same remaining gap.

In [5]:
comparison = (
    horizon[horizon.arm.isin(["measured", "posterior"])]
    .pivot_table(index="dataset", columns="arm", values="k_epi", aggfunc="median")
    .dropna()
)
comparison["ratio"] = (comparison.posterior / comparison.measured).round(2)
comparison.round(2).to_string()

'arm      measured  posterior  ratio\ndataset                            \nDHFR        -3.83       3.85  -1.01\nGB1          4.34       4.17   0.96\nParD3        1.03       0.92   0.90\nT7           1.83       4.46   2.43\nTEV          1.40       2.37   1.70\nTrpB3D       1.74       0.93   0.54\nTrpB3G       1.74       1.80   1.03\nTrpB3I       6.03       8.38   1.39\nTrpB4        2.39       0.29   0.12'

In [6]:
print(json.dumps(json.loads((ROOT / "summary.json").read_text()), indent=2))

{
  "seed": 20260822,
  "n_rows": 205,
  "median_k_epi": {
    "measured": 2.129755355381136,
    "posterior": 3.2696082121876664
  },
  "negative_k_epi_fraction": {
    "measured": 0.0625,
    "posterior": 0.13333333333333333
  },
  "median_eps_plus": {
    "measured": -0.2367050240533135,
    "posterior": -0.06982906911731958
  }
}


## What this licenses in the paper

A claim SLIP cannot make from its own framework. Their difficulty parameter now
has a measured reference value, their untuned models are quantifiably too
smooth, and their definition degenerates on the majority of real assays because
it presumes beneficial singles exist to combine.

The remaining comparison worth running is SLIP itself as a fourth arm in
`run_topology_benchmark.py`, asking whether any choice of horizon reproduces
measured local optima counts and Walsh spectra. If none does, difficulty and
realism are separate axes.